# Create Iceberg table for Bronze and Silver layer
This Notebook used for initialize namespaces and define data language (DDL) for Bronze and Silver layer

In [ ]:
from pyspark.sql import SparkSession

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("FlightBooking_Create_Tables") \
    .getOrCreate()

# Create Database/Namespace in Iceberg
spark.sql("CREATE NAMESPACE IF NOT EXISTS demo.bronze")
spark.sql("CREATE NAMESPACE IF NOT EXISTS demo.silver")
spark.sql("CREATE NAMESPACE IF NOT EXISTS demo.gold")

print("Initialize namespaces successfully!")

26/07/21 07:35:15 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


✅ Khởi tạo namespaces thành công!


## Reset: Delete all databases

In [ ]:
# # Delete all tables in database bronze and silver
# spark.sql("DROP DATABASE IF EXISTS demo.bronze CASCADE")
# spark.sql("DROP DATABASE IF EXISTS demo.silver CASCADE")
# spark.sql("DROP DATABASE IF EXISTS demo.gold CASCADE")

# print("Delete databases successfully!")

In [14]:
%%sql

SHOW DATABASES

namespace
bronze
silver
gold


## Reset: Delete the old table if exists

In [ ]:
# Delete Silver tables
spark.sql("DROP TABLE IF EXISTS demo.silver.pnr_records_clean")
spark.sql("DROP TABLE IF EXISTS demo.silver.passengers_clean")
spark.sql("DROP TABLE IF EXISTS demo.silver.flight_segments_clean")
spark.sql("DROP TABLE IF EXISTS demo.silver.tickets_clean")
spark.sql("DROP TABLE IF EXISTS demo.silver.payments_clean")
spark.sql("DROP TABLE IF EXISTS demo.silver.booking_events_clean")

# Delete Bronze tables
spark.sql("DROP TABLE IF EXISTS demo.bronze.pnr_records")
spark.sql("DROP TABLE IF EXISTS demo.bronze.passengers")
spark.sql("DROP TABLE IF EXISTS demo.bronze.flight_segments")
spark.sql("DROP TABLE IF EXISTS demo.bronze.tickets")
spark.sql("DROP TABLE IF EXISTS demo.bronze.payments")
spark.sql("DROP TABLE IF EXISTS demo.bronze.booking_events")

print("Reset tables successfully!")

✅ Reset các bảng thành công!


## 1. DDL: Bronze Layer (Raw CDC logs - Update right data types)

In [ ]:
# Khai báo bảng Bronze khớp kiểu dữ liệu với file Parquet từ Kafka Connect S3 Sink
# Declare Bronze tables match type with file Parquet form Kafka Connect S3 Sink
spark.sql("""
CREATE TABLE IF NOT EXISTS demo.bronze.pnr_records (
    pnr_id           STRING,
    booking_channel  STRING,
    booking_status   STRING,
    created_at       TIMESTAMP,
    updated_at       TIMESTAMP,
    __deleted        STRING,
    __op             STRING,
    __table          STRING,
    __source_ts_ms   BIGINT,
    event_date       STRING
) USING iceberg
PARTITIONED BY (event_date)
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS demo.bronze.passengers (
    passenger_id     INT,
    pnr_id           STRING,
    first_name       STRING,
    last_name        STRING,
    email            STRING,
    passport_number  STRING,
    created_at       TIMESTAMP,
    updated_at       TIMESTAMP,
    __deleted        STRING,
    __op             STRING,
    __table          STRING,
    __source_ts_ms   BIGINT,
    event_date       STRING
) USING iceberg
PARTITIONED BY (event_date)
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS demo.bronze.flight_segments (
    segment_id       INT,
    pnr_id           STRING,
    origin_airport   STRING,
    dest_airport     STRING,
    flight_date      DATE,
    airline_code     STRING,
    flight_number    STRING,
    created_at       TIMESTAMP,
    updated_at       TIMESTAMP,
    __deleted        STRING,
    __op             STRING,
    __table          STRING,
    __source_ts_ms   BIGINT,
    event_date       STRING
) USING iceberg
PARTITIONED BY (event_date)
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS demo.bronze.tickets (
    ticket_number    STRING,
    passenger_id     INT,
    fare_class       STRING,
    ticket_status    STRING,
    created_at       TIMESTAMP,
    updated_at       TIMESTAMP,
    __deleted        STRING,
    __op             STRING,
    __table          STRING,
    __source_ts_ms   BIGINT,
    event_date       STRING
) USING iceberg
PARTITIONED BY (event_date)
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS demo.bronze.payments (
    payment_id       STRING,
    pnr_id           STRING,
    payment_method   STRING,
    amount           DECIMAL(12,2),
    currency         STRING,
    payment_status   STRING,
    created_at       TIMESTAMP,
    updated_at       TIMESTAMP,
    __deleted        STRING,
    __op             STRING,
    __table          STRING,
    __source_ts_ms   BIGINT,
    event_date       STRING
) USING iceberg
PARTITIONED BY (event_date)
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS demo.bronze.booking_events (
    event_id         INT,
    pnr_id           STRING,
    event_type       STRING,
    created_at       TIMESTAMP,
    __deleted        STRING,
    __op             STRING,
    __table          STRING,
    __source_ts_ms   BIGINT,
    event_date       STRING
) USING iceberg
PARTITIONED BY (event_date)
""")

print("DDL for 6 Bronze tables done!")

✅ Khai báo DDL cho toàn bộ 6 bảng Bronze hoàn tất!


## 2. DDL: Silver Layer (Cleaned and Merged tables)

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS demo.silver.pnr_records_clean (
    pnr_id           STRING NOT NULL,
    booking_channel  STRING,
    booking_status   STRING,
    created_at       TIMESTAMP,
    updated_at       TIMESTAMP
) USING iceberg
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS demo.silver.passengers_clean (
    passenger_id     INT NOT NULL,
    pnr_id           STRING NOT NULL,
    first_name       STRING,
    last_name        STRING,
    email            STRING,
    passport_number  STRING,
    created_at       TIMESTAMP,
    updated_at       TIMESTAMP
) USING iceberg
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS demo.silver.flight_segments_clean (
    segment_id       INT NOT NULL,
    pnr_id           STRING NOT NULL,
    origin_airport   STRING,
    dest_airport     STRING,
    flight_date      DATE,
    airline_code     STRING,
    flight_number    STRING,
    created_at       TIMESTAMP,
    updated_at       TIMESTAMP
) USING iceberg
PARTITIONED BY (months(flight_date))
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS demo.silver.tickets_clean (
    ticket_number    STRING NOT NULL,
    passenger_id     INT NOT NULL,
    fare_class       STRING,
    ticket_status    STRING,
    created_at       TIMESTAMP,
    updated_at       TIMESTAMP
) USING iceberg
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS demo.silver.payments_clean (
    payment_id       STRING NOT NULL,
    pnr_id           STRING NOT NULL,
    payment_method   STRING,
    amount           DECIMAL(12,2),
    currency         STRING,
    payment_status   STRING,
    created_at       TIMESTAMP,
    updated_at       TIMESTAMP
) USING iceberg
PARTITIONED BY (months(created_at))
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS demo.silver.booking_events_clean (
    event_id         INT NOT NULL,
    pnr_id           STRING NOT NULL,
    event_type       STRING,
    created_at       TIMESTAMP
) USING iceberg
PARTITIONED BY (months(created_at))
""")

print("DDL for 6 Silver tables done!")

✅ Khai báo DDL cho toàn bộ 6 bảng Silver hoàn tất!


In [11]:
%%sql

SHOW TABLES FROM demo.bronze

namespace,tableName,isTemporary
bronze,booking_events,False
bronze,flight_segments,False
bronze,passengers,False
bronze,payments,False
bronze,pnr_records,False
bronze,tickets,False


In [12]:
%%sql

SHOW TABLES FROM demo.silver

namespace,tableName,isTemporary
silver,booking_events_clean,False
silver,flight_segments_clean,False
silver,passengers_clean,False
silver,payments_clean,False
silver,pnr_records_clean,False
silver,tickets_clean,False


In [13]:
%%sql

SHOW TABLES FROM demo.gold

namespace,tableName,isTemporary
